In [ ]:
import coflandscaper as cl
from pathlib import Path
from pymatgen.core import Structure

/Users/gregorlauter/Documents/PhD/2d-cof-mapper/COF-Landscaper-dev/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [2]:
TOPOLOGY = "sql"
COF_NAME = "PDI-BT-COF-same"
MODE = "both"

In [3]:
builder = cl.BuildCOF2D()
outputs = builder.build_same_and_flip(
    topo=TOPOLOGY,
    cof_name=COF_NAME,
)

>>> == Min RMSD of (node type: 0, node bb: node_same_v_small_matrix): 2.52E-01
>>> Pre-location at node slot 0, (node type: 0, node bb: node_same_v_small_matrix), RMSD: 2.52E-01
>>> Topology optimization starts.
>>> MESSAGE: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
>>> SUCCESS: True
>>> ITER: 6
>>> OBJ: 0.000
>>> Location at node slot 0, (node type: 0, node bb: node_same_v_small_matrix), RMSD: 1.22E-02
>>> Start placing edges.
/Users/gregorlauter/Documents/PhD/2d-cof-mapper/COF-Landscaper-dev/.venv/lib/python3.12/site-packages/scipy/spatial/transform/_rotation.py:2586: UserWarning: Optimal rotation is not uniquely or poorly defined for the given sets of vectors.
  q, rssd, sensitivity = backend.align_vectors(a, b, weights, return_sensitivity)
>>> Start finding bonds in generated framework.
>>> Start finding bonds in building blocks.
>>> Start finding bonds between building blocks.
>>> Start making Framework instance.
>>> Construction of framework done.
>>> == Min RMSD of (n

In [ ]:
single_layer_dir = Path(COF_NAME) / f"1_{COF_NAME}_single_layer"
same_unopt_cif = Path(outputs["same_unopt"])
flip_unopt_cif = Path(outputs["flip_unopt"])
double_unopt_cif = single_layer_dir / f"{COF_NAME}_double_unopt.cif"

cl.combine_single_layers(
    layer_a_cif=same_unopt_cif,
    layer_b_cif=flip_unopt_cif,
    output_cif=double_unopt_cif,
    interlayer_distance=15.0,
)

a_atoms = len(Structure.from_file(str(same_unopt_cif)))
b_atoms = len(Structure.from_file(str(flip_unopt_cif)))
double_atoms = len(Structure.from_file(str(double_unopt_cif)))
if double_atoms != a_atoms + b_atoms:
    raise ValueError(
        f"Unexpected atom count in combined CIF: {double_atoms} != {a_atoms + b_atoms}"
    )
print("double_unopt:", double_unopt_cif)
print("atom_count_check:", double_atoms)

double_unopt: PDI-BT-COF-same/1_PDI-BT-COF-same_single_layer/PDI-BT-COF-same_double_unopt.cif
atom_count_check: 376


/Users/gregorlauter/Documents/PhD/2d-cof-mapper/COF-Landscaper-dev/.venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3112: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


In [ ]:
preopt = cl.MaceOpt(
    fmax=0.7,
    device="cpu",
)
preopt.run_preopt(
    cof_name=COF_NAME,
    fix_z=True,
    input_path=f"{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_double_unopt.cif",
    output_path=f"{COF_NAME}/1_{COF_NAME}_single_layer/{COF_NAME}_preopt.cif",
)

In [5]:
double_preopt_cif = single_layer_dir / f"{COF_NAME}_preopt.cif"
matrix = cl.CreateMatrix(explicit_bilayer=True)
matrix.run(
    cof_name=COF_NAME,
    topo=TOPOLOGY,
    mode=MODE,
    input_cif=str(double_preopt_cif),
)
print("matrix input:", double_preopt_cif)

matrix input: PDI-BT-COF-same/1_PDI-BT-COF-same_single_layer/PDI-BT-COF-same_preopt.cif


In [ ]:
landscape = cl.Landscape()
landscape.run_mode(cof_name=COF_NAME, mode=MODE, show=True, rel_energy_max=None)

In [ ]:
analyzer = cl.AnalyzeStacking()
analyzer.analyze(cof_name=COF_NAME, mode=MODE)

In [ ]:
pxrd = cl.PXRD()
pxrd.plot_sim_vs_exp(
    cof_name=COF_NAME,
    mode=MODE,
    xlim=(1.5, 30.0),
)